# Complementario: cómo armamos el dataset del lab "Destilar Countdown"

Este notebook genera `countdown_precalculo.json`, el **dataset ya armado** que usa el lab. No hace falta correrlo:
es para quien quiera ver cómo se hizo, o generarlo de nuevo (por ejemplo, con otro profesor o más problemas).

Qué contiene:
- **entrenamiento:** problemas de 3 números con una respuesta del profesor (Qwen3-4B-Instruct-2507, T=0,7);
- **evaluación:** 100 problemas de 3 números con la respuesta greedy del profesor y del alumno (Qwen3-0.6B);
- **generalización:** 50 problemas de **4** números con la respuesta greedy del alumno (para la pregunta final).

Correr en Colab con T4. Va por **etapas** que se guardan en disco: si Colab se corta, vuelvan a ejecutar todo y
retoma donde quedó (mientras no se borre el entorno). Tiempo: **cerca de una hora** (casi todo es la etapa 1, el profesor).

Antes medimos la brecha (nov 2026, T4): en 3 números el alumno acierta ~38% y el profesor ~93%; en 4 números el
profesor se corta el 62% de las veces sin terminar, por eso el lab usa 3.

**Para probar:** cambien `PROFESOR`, `N_ENTRENAMIENTO` o `TEMPERATURA` en la celda de configuración, generen su
propio JSON y úsenlo en el lab (sección 3, subiéndolo a mano). Si cambian el profesor, que sea un Qwen3: la opción
"probabilidades" del lab necesita que profesor y alumno compartan vocabulario.

In [ ]:
!pip install -q -U transformers accelerate datasets

In [ ]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import json
import random
import time

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import re

INSTRUCCION = ("Using the numbers {nums}, create an equation that equals {target}. You can use basic arithmetic "
               "operations (+, -, *, /) and each number must be used exactly once. Think step by step briefly, then "
               "write the final equation inside <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.")

_PERMITIDOS = re.compile(r"^[\d+\-*/().\s]+$")


def armar_prompt(p):
    return INSTRUCCION.format(nums=p["nums"], target=p["target"])


def mensajes(p):
    return [{"role": "user", "content": armar_prompt(p)}]


def extraer_respuesta(texto):
    """La última ecuación entre <answer> y </answer>, o None si no hay."""
    encontradas = re.findall(r"<answer>(.*?)</answer>", texto, re.S)
    return encontradas[-1].strip() if encontradas else None


def verificar(texto, p):
    """Tres notas, de menos a más exigente:
    formato: hay una respuesta entre <answer> tags;
    reglas:  la ecuación solo usa + - * / ( ) y cada número dado exactamente una vez;
    correcta: además, da el número objetivo."""
    ecuacion = extraer_respuesta(texto)
    nota = {"formato": ecuacion is not None, "reglas": False, "correcta": False, "ecuacion": ecuacion}
    if not ecuacion or not _PERMITIDOS.match(ecuacion) or "**" in ecuacion or "//" in ecuacion:
        return nota
    usados = sorted(int(n) for n in re.findall(r"\d+", ecuacion))
    if usados != sorted(p["nums"]):
        return nota
    nota["reglas"] = True
    try:
        valor = eval(ecuacion, {"__builtins__": None}, {})  # solo dígitos, operadores y paréntesis (regex de arriba)
    except Exception:
        return nota
    nota["correcta"] = abs(float(valor) - p["target"]) < 1e-5
    return nota


def plantilla(tok, msgs):
    """Plantilla de chat de Qwen3 sin modo thinking (si la plantilla no acepta el argumento, sin él)."""
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
ALUMNO = "Qwen/Qwen3-0.6B"
PROFESOR = "Qwen/Qwen3-4B-Instruct-2507"
N_ENTRENAMIENTO = 1000   # problemas de 3 números para entrenar
N_EVALUACION = 100       # problemas de 3 números para evaluar (nunca se entrenan)
N_GENERALIZACION = 50    # problemas de 4 números (solo para la pregunta final)
TEMPERATURA = 0.7
MAX_TOKENS = 768         # en 3 números el profesor usa ~330 tokens y se corta el 1% a 1024
LOTE_PROFESOR = 24
LOTE_ALUMNO = 32
print("Dispositivo:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

In [ ]:
ds = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
idx = list(range(len(ds)))
random.Random(2026).shuffle(idx)   # misma semilla que medir_brecha
tres, cuatro = [], []
for i in idx:
    fila = {"target": int(ds[i]["target"]), "nums": [int(n) for n in ds[i]["nums"]]}
    (tres if len(fila["nums"]) == 3 else cuatro).append(fila)
    if len(tres) >= N_EVALUACION + N_ENTRENAMIENTO and len(cuatro) >= N_GENERALIZACION:
        break
EVA, ENT, GEN = tres[:N_EVALUACION], tres[N_EVALUACION:N_EVALUACION + N_ENTRENAMIENTO], cuatro[:N_GENERALIZACION]
print(len(ENT), "para entrenar ·", len(EVA), "para evaluar ·", len(GEN), "de 4 números")

In [ ]:
def cargar(nombre):
    tok = AutoTokenizer.from_pretrained(nombre)
    tok.padding_side = "left"
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float16 if device == "cuda" else torch.float32)
    return tok, modelo.to(device).eval()


@torch.no_grad()
def generar(tok, modelo, problemas, temperatura, lote):
    """Una respuesta por problema, ordenando por largo del prompt para desperdiciar menos relleno."""
    orden = sorted(range(len(problemas)), key=lambda k: len(armar_prompt(problemas[k])))
    salida, total, t0 = [None] * len(problemas), 0, time.time()
    for i in range(0, len(orden), lote):
        grupo = orden[i:i + lote]
        ids = tok([plantilla(tok, mensajes(problemas[k])) for k in grupo], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(device)
        extra = dict(do_sample=True, temperature=temperatura, top_p=0.8, top_k=20) if temperatura > 0 else dict(do_sample=False)
        out = modelo.generate(**ids, max_new_tokens=MAX_TOKENS, pad_token_id=tok.pad_token_id, **extra)
        for k, fila in zip(grupo, out[:, ids["input_ids"].shape[1]:]):
            texto = tok.decode(fila, skip_special_tokens=True)
            n = int((fila != tok.pad_token_id).sum())
            nota = verificar(texto, problemas[k])
            salida[k] = {"texto": texto, "ntok": n, "cortada": n >= MAX_TOKENS,
                         **{c: nota[c] for c in ("formato", "reglas", "correcta")}}
            total += n
        print(f"  {min(i + lote, len(orden))}/{len(orden)} · {total / (time.time() - t0):.0f} tokens/s", end="\r")
    print()
    return salida


def etapa(archivo, funcion):
    if os.path.exists(archivo):
        print("ya estaba:", archivo)
        return json.load(open(archivo))
    t0 = time.time()
    r = funcion()
    json.dump(r, open(archivo, "w"), ensure_ascii=False)
    print(f"lista en {(time.time() - t0) / 60:.1f} min → {archivo}")
    return r


def informe(nombre, rs):
    print(f"{nombre:38s} correcta {100 * np.mean([r['correcta'] for r in rs]):5.1f}%  "
          f"formato {100 * np.mean([r['formato'] for r in rs]):5.1f}%  tokens {np.mean([r['ntok'] for r in rs]):4.0f}  "
          f"cortadas {100 * np.mean([r['cortada'] for r in rs]):4.1f}%")

In [ ]:
#@title Etapa 1: el profesor (la más larga)
def _profesor():
    tok, modelo = cargar(PROFESOR)
    r = {"entrenamiento": generar(tok, modelo, ENT, TEMPERATURA, LOTE_PROFESOR),
         "evaluacion": generar(tok, modelo, EVA, 0.0, LOTE_PROFESOR)}
    del modelo
    torch.cuda.empty_cache() if device == "cuda" else None
    return r


PROF = etapa("etapa1_profesor.json", _profesor)
informe("profesor · entrenamiento (T=0,7)", PROF["entrenamiento"])
informe("profesor · evaluación (greedy)", PROF["evaluacion"])

In [ ]:
#@title Etapa 2: el alumno antes de entrenar
def _alumno():
    tok, modelo = cargar(ALUMNO)
    r = {"evaluacion": generar(tok, modelo, EVA, 0.0, LOTE_ALUMNO),
         "generalizacion": generar(tok, modelo, GEN, 0.0, LOTE_ALUMNO)}
    del modelo
    torch.cuda.empty_cache() if device == "cuda" else None
    return r


ALUM = etapa("etapa2_alumno.json", _alumno)
informe("alumno · evaluación 3 números (greedy)", ALUM["evaluacion"])
informe("alumno · 4 números (greedy)", ALUM["generalizacion"])

In [ ]:
#@title Etapa 3: armar el archivo final
DATOS = {
    "meta": {"modelo_profesor": PROFESOR, "modelo_alumno": ALUMNO, "temperatura": TEMPERATURA, "max_tokens": MAX_TOKENS,
             "instruccion": INSTRUCCION, "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
             "fecha": time.strftime("%Y-%m-%d")},
    "entrenamiento": [dict(p, profesor=r) for p, r in zip(ENT, PROF["entrenamiento"])],
    "evaluacion": [dict(p, profesor=a, alumno=b) for p, a, b in zip(EVA, PROF["evaluacion"], ALUM["evaluacion"])],
    "generalizacion": [dict(p, alumno=b) for p, b in zip(GEN, ALUM["generalizacion"])],
}
json.dump(DATOS, open("countdown_precalculo.json", "w"), ensure_ascii=False)
print(f"countdown_precalculo.json: {os.path.getsize('countdown_precalculo.json') / 1e6:.1f} MB")
print("Entrenamiento utilizable (respuesta correcta del profesor):",
      sum(r["correcta"] for r in PROF["entrenamiento"]), "de", len(ENT))
print("\nDescárguenlo (panel de archivos → clic derecho → Descargar) y súbanlo en la sección 3 del lab.")